# BA283 Fidelity Ablation — Colab T4 (single venue)

Runs the Core-4 OFAT ablation cells **all on Colab GPU** so every cell shares one
baseline (`B0`). Scores with the `eval/ablation/` harness, writes a verdict table,
and `rclone copy`s every result to Drive (survives a runtime kill).

Cells: **B0** (reference) · **C1** (hidden 64) · **C4** (88.2 kHz) · **C5** (9-pt gain grid).
C7 (target floor) runs locally — see `docs/colab_ablation_runbook.md`.

Runtime → Change runtime type → **T4 GPU**, then run top to bottom. Prereq: the
ablation subsets are already on Drive under `ba283_ablation/` (see the runbook).

## 1. Install rclone + upload `rclone.conf`
No mount: subsets are copied to `/content` later so training never reads through FUSE. Upload your `rclone.conf` (find it via `rclone config file`); it holds the Drive token, written straight to disk.

In [ ]:
import os, subprocess
from google.colab import files

# install rclone (no-op if present)
if subprocess.run(['which','rclone'],capture_output=True).returncode:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash', shell=True, check=True)

# upload rclone.conf (token written to disk, not echoed)
cfg = os.path.expanduser('~/.config/rclone/rclone.conf')
os.makedirs(os.path.dirname(cfg), exist_ok=True)
up = files.upload()  # pick your rclone.conf
assert up, 'no file uploaded'
open(cfg,'wb').write(next(iter(up.values())))
REMOTE = subprocess.run(['rclone','listremotes'],capture_output=True,text=True).stdout.split()[0]
print('remote:', REMOTE)

## 2. Clone the repo + deps + GPU check
Paste a GitHub token with read access (kept out of output via getpass). `%cd` so `train`/`eval` import and `python -m train.train` resolves.

In [ ]:
import getpass, subprocess
TOKEN = getpass.getpass('GitHub token: ')
subprocess.run(['git','clone','-q','https://x-access-token:%s@github.com/gustavokch/ba283-vst3.git'%TOKEN,'/content/ba283'])
%cd /content/ba283
!git pull -q
!pip -q install soundfile pandas tensorboard scipy
import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime type to T4 GPU'
print(torch.cuda.get_device_name(0))

## 3. Config — controls + subset layout
Fixed controls from the ablation design (seed, epoch budget, early-stop). `SUBSETS` maps each cell to its gdrive subdir + dry/wet sample rate. B0 and C1 share `base44k`; C4 is 88.2 kHz; C5 is the 9-pt gain grid.

In [ ]:
SEED, EPOCHS, PATIENCE = 745603, 60, 15
TRAINED_GAINS = {0.0, 0.25, 0.5, 0.75, 1.0}
ABL = REMOTE + 'ba283_ablation/'          # gdrive root for subsets + results
SUBSETS = {                               # cell -> (gdrive subdir, dry/wet SR)
    'base44k': (ABL + 'base44k', 44100),  # B0 + C1 share this
    'c4_88k':  (ABL + 'c4_88k', 88200),
    'c5_9pt':  (ABL + 'c5_9pt', 44100),
}

## 4. Stage subsets to local disk + remap manifests
Reuses `train.stage`: `rclone copy` each subset to `/content/<key>` (no mount; asserts `manifest.csv`), then rewrite manifest audio paths to those local files.

In [ ]:
from pathlib import Path
from train.stage import stage_rclone, remap_manifests

def stage_cell(key, names=('train', 'val')):
    remote, _sr = SUBSETS[key]
    dest = f'/content/{key}'
    stage_rclone(remote, dest)                              # rclone copy, asserts manifest.csv
    return remap_manifests(dest, names=names, out_dir=dest)  # -> {name: csv path}

MAN = {k: stage_cell(k) for k in SUBSETS}
print(MAN)

## 5. B0 — reference (hidden 32, 44.1 kHz)
The single-venue baseline every other cell is scored against.

In [ ]:
!python -m train.train \
  --train-manifest /content/base44k/train.csv --val-manifest /content/base44k/val.csv \
  --epochs 60 --patience 15 --hidden 32 --conditioning film --device cuda \
  --checkpoint-dir ckpt_b0 --log-dir logs_b0 --seed 745603

## 6. C1 — capacity (hidden 64, 44.1 kHz)
Same subset as B0, wider LSTM. Tests the capacity ceiling.

In [ ]:
!python -m train.train \
  --train-manifest /content/base44k/train.csv --val-manifest /content/base44k/val.csv \
  --epochs 60 --patience 15 --hidden 64 --conditioning film --device cuda \
  --checkpoint-dir ckpt_c1 --log-dir logs_c1 --seed 745603

## 7. C4 — aliasing (hidden 32, 88.2 kHz)
Trains on the 88.2 kHz subset. Eval downsamples the model output to 44.1 kHz (matched anti-alias) so the ESR is apples-to-apples with B0, and reports a high-band aliasing error.

In [ ]:
!python -m train.train \
  --train-manifest /content/c4_88k/train.csv --val-manifest /content/c4_88k/val.csv \
  --epochs 60 --patience 15 --hidden 32 --conditioning film --device cuda \
  --checkpoint-dir ckpt_c4 --log-dir logs_c4 --seed 745603

## 8. C5 — gain density (hidden 32, 9-pt grid)
Trains on the 5 trained gains; the val manifest carries all 9 gains so held-out-gain ESR measures FiLM interpolation at the 4 unseen gains.

In [ ]:
!python -m train.train \
  --train-manifest /content/c5_9pt/train.csv --val-manifest /content/c5_9pt/val.csv \
  --epochs 60 --patience 15 --hidden 32 --conditioning film --device cuda \
  --checkpoint-dir ckpt_c5 --log-dir logs_c5 --seed 745603

## 9. Score all cells + verdict table + save to Drive
Scores each checkpoint with the `eval/ablation/` harness, builds the verdict table (ΔdB vs B0, held-out-gain, ✅/—), and `rclone copy`s the results to `ba283_ablation/results` so they survive a runtime kill.

In [ ]:
import json, os, subprocess
import numpy as np
from eval.ablation.run_checkpoint_eval import run_cell
from eval.ablation.report import render_table

TARGET = 0.01  # redesign target (~ -20 dB); decision-rule reference
SPECS = [  # (cell, ckpt, hidden, val manifest, sr_train, sr_eval)
    ('B0', 'ckpt_b0/best.pth', 32, '/content/base44k/val.csv', None, None),
    ('C1', 'ckpt_c1/best.pth', 64, '/content/base44k/val.csv', None, None),
    ('C4', 'ckpt_c4/best.pth', 32, '/content/c4_88k/val.csv', 88200, 44100),
    ('C5', 'ckpt_c5/best.pth', 32, '/content/c5_9pt/val.csv', None, None),
]
results = []
for name, ckpt, hidden, man, srt, sre in SPECS:
    r = run_cell(ckpt, man, hidden=hidden, conditioning='film', cell=name,
                 trained_gains=TRAINED_GAINS, sr_train=srt, sr_eval=sre)
    results.append(r)
    print('%-3s val_esr=%.4f (%.1f dB) held-out=%s' %
          (name, r['val_esr'], 10*np.log10(max(r['val_esr'], 1e-12)),
           r['held_out_gain_esr']))

base = next(r['val_esr'] for r in results if r['cell'] == 'B0')
for r in results:
    r['baseline_esr'] = base
table = render_table(results, target=TARGET)
print('\n' + table)

os.makedirs('/content/abl_save', exist_ok=True)
json.dump(results, open('/content/abl_save/ablation_results.json', 'w'), indent=2)
open('/content/abl_save/ablation_table.md', 'w').write(table)
subprocess.run(['rclone', 'copy', '/content/abl_save', ABL + 'results', '-P'], check=True)
print('saved -> %sresults' % ABL)

Next: run **C7** (target floor) locally and confirm the winning axis on 2-3 seeds before it graduates to the redesign plan — see `docs/colab_ablation_runbook.md` §7-8.